# A well-diagnosed wrong number

Every other notebook in this series asks whether a model deserves belief: does the procedure
cover, does the posterior predict, does the graph survive refutation. All of it is downstream
of a question nobody runs: **did the units arrive in the arms the design put them in?**

A 51/49 split where 50/50 was asked for is invisible in a summary table and impossible to see
by eye. On a hundred thousand units it is a ten-sigma event, and whatever caused it — a
filter, a retry, a logging drop that hit one arm — is very unlikely to be independent of the
outcome. Fitting a beautiful model on top of it produces a well-diagnosed wrong number.

Three checks, each the challenge to an assumption `design` already names.

In [ ]:
import numpy as np

from axiom.design import ASSUMPTIONS, ArmAllocation, assign
from axiom.diagnose import (
    SRM_ALPHA, ArmCount, BalanceCheck, BalanceTest, Delivery, DeliveryReport, DeliveryRow,
    SampleRatio, arm_counts, balance, check_delivery, delivery, sample_ratio,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way

enable();  # every axiom result renders itself

split = ArmAllocation.equal("control", "treated")
for name in ("random_assignment", "exposure_known"):
    a = ASSUMPTIONS[name]
    print(f"{name}: {a.statement}\n  challenged by: {a.challenged_by}")

## 1. The sample-ratio check

A chi-square goodness of fit of the realized arm counts against the allocation's shares. The
alpha is `0.001`, not `0.05`: this check runs on every experiment, and at 5 % one experiment
in twenty fails it with nothing wrong, which is how a check stops being read.

In [ ]:
print("SRM_ALPHA =", SRM_ALPHA)
rows = []
for label, counts in (("clean 10k", {"control": 5012, "treated": 4988}),
                      ("51/49 on 100k", {"control": 51000, "treated": 49000}),
                      ("one arm short", {"control": 5000, "treated": 4700})):
    check: SampleRatio = sample_ratio(counts, split)
    rows.append([label, str(counts), f"{check.chi_square:.2f}", f"{check.p_value:.3g}",
                 "MISMATCH" if check.mismatch else "ok", check.assumption().state])
table(rows, headers=("split", "counts", "chi-square", "p", "verdict", "random_assignment"),
      title=f"sample ratio at alpha {SRM_ALPHA}")

caught = sample_ratio({"control": 51000, "treated": 49000}, split)
worst: ArmCount = max(caught.arms, key=lambda a: abs(a.deviation))
print(f"\nworst arm {caught.worst_arm}: {worst.observed} against {worst.expected:.0f} expected"
      f" ({worst.deviation:+.0f}, realized share {caught.realized_share(worst.arm):.4f})")
print(caught.ledger_line().statement[:120], "...")

The three-arm case shows why the alpha matters. A high arm 6 % light is `p = 0.0022` — enough
to worry at 5 %, not enough to stop a readout at 0.001. Both answers are defensible; only one
of them can be the default for a check that runs every week.

In [ ]:
tri = ArmAllocation(arms=("control", "low", "high"), shares=(0.5, 0.25, 0.25))
observed = {"control": 5000, "low": 2500, "high": 2300}
table([[f"{a:g}", f"{sample_ratio(observed, tri, alpha=a).p_value:.4g}",
        str(sample_ratio(observed, tri, alpha=a).mismatch)] for a in (0.05, 0.01, SRM_ALPHA)],
      headers=("alpha", "p", "mismatch"), title=f"the same counts {observed}")
print("counted from a column of labels:", arm_counts(["control", "treated", "control"], split))

## 2. Delivery: equal rates, not equal counts

Two arms can be exactly the right size and still not be comparable, if one of them was
actually *reached* far less often. `delivery` tests the arm-by-exposed table: the tail
probability of a gap this large if every arm were reached at one common rate.

In [ ]:
rows = []
for label, exposed in (("even", {"control": 4500, "treated": 4480}),
                       ("treatment under-delivered", {"control": 4900, "treated": 3000})):
    check: Delivery = delivery({"control": 5000, "treated": 5000}, exposed)
    rates = ", ".join(f"{r.arm} {r.rate:.3f}" for r in check.arms)
    rows.append([label, rates, f"{check.differential:.3f}", f"{check.p_value:.3g}",
                 "DIFFERENTIAL" if check.differential_delivery else "ok",
                 check.assumption().state])
table(rows, headers=("case", "rates", "gap", "p", "verdict", "exposure_known"))

uneven = delivery({"control": 5000, "treated": 5000}, {"control": 4900, "treated": 3000})
first: DeliveryRow = uneven.arms[0]
print(f"\n{first.arm}: {first.exposed} of {first.assigned} reached ({first.rate:.3f});"
      f" overall {uneven.overall_rate:.3f}")
print("An ITT contrast between these arms is not the contrast anybody wrote down.")

## 3. Balance on the covariates somebody recorded

A one-way F test per covariate on the realized arms, Holm-corrected because the question is
whether *any* covariate is out of balance. This is the weakest of the three: it can only look
at what was recorded, so a clean balance table says nothing about the covariates nobody wrote
down.

In [ ]:
rng = np.random.default_rng(0)
units = tuple(f"u{i:05d}" for i in range(4000))
assigned = assign(units, split, salt="NW-14")
covariates = {
    "age": rng.uniform(20, 60, len(units)),
    "pre_outcome": rng.normal(size=len(units)),
    "tenure": rng.normal(size=len(units)) + 0.35 * assigned.arm_of,   # planted imbalance
    "flat": np.ones(len(units)),                                      # nothing to be off about
}
check: BalanceCheck = balance(covariates, assigned)
table([[t.covariate, f"{t.smd:.4f}", f"{t.f_statistic:.2f}", f"{t.p_value:.3g}",
        f"{t.adjusted_p:.3g}", str(t.imbalanced)] for t in check.tests],
      headers=("covariate", "smd", "F", "p", f"{check.correction}-adjusted", "imbalanced"))
print("skipped (constant, nothing to be imbalanced about):", check.skipped)
print("imbalanced:", check.imbalanced, "| random_assignment:", check.assumption().state)

## One call, one verdict, and what it will not say

`check_delivery` runs every check the arguments support and reports which ones actually ran —
what is not supplied is never silently passed. The verdict is `identified` or `blocked` and
never `downgraded`: an assumption can license a transfer across a facet difference, and
nothing licenses a readout past a split that did not happen.

In [ ]:
clean: DeliveryReport = check_delivery({"control": 5012, "treated": 4988}, split)
print(clean.summary())
print("  route:", clean.verdict().route, "| checks run:", clean.checks_run)

full = check_delivery(assigned, exposed={"control": 1900, "treated": 1200}, covariates=covariates)
print("\n" + full.summary())
print("  failures:", full.failures)
table([[line.kind, line.assumption.name, line.assumption.state] for line in full.ledger()],
      headers=("ledger line", "assumption", "state"), title="what the readout carries")
print("\ndowngraded is not on the menu:", full.verdict().status, "|",
      "assumptions on the verdict:", full.verdict().assumptions)

## What this notebook decided

- The first diagnostic is not about the model. A 51/49 split is a ten-sigma event that no
  summary table shows, and it invalidates everything downstream of it.
- The alpha for a check that runs every week is `0.001`. A check that cries wolf weekly is a
  check nobody reads, and that is a worse failure than the one it was guarding against.
- Equal arm sizes are not equal delivery. Arms reached at 98 % and 60 % are not comparable
  however even their counts are.
- **A passing check leaves the assumption `unverified`, not `satisfied`.** Absence of evidence
  of a broken randomizer is not evidence of a working one, and the ledger line says which of
  the two it is carrying. Only a failure moves an assumption to `violated`.